# Практика 3. От покупок к первому прогнозу

**Финансовый университет · Технологии финансовой аналитики**

Сложим покупки каждого клиента, присоединим известный исход и обучим логистическую регрессию.

Положите `practice_purchases.csv` и `practice_target.csv` рядом с ноутбуком.
В Colab загрузите оба файла в панель «Файлы». Выполняйте ячейки сверху вниз.

## 1. Groupby: складываем покупки одного клиента

В `practice_purchases.csv` **одна строка — одна покупка**.
У одного клиента может быть несколько покупок. Пока посмотрим только на первые шесть строк.

In [ ]:
import pandas as pd

purchases = pd.read_csv("practice_purchases.csv")
purchases.head(6)

Представьте, что раскладываем чеки по трём конвертам. На каждом конверте написан ID клиента.
Чеки `C001` кладём вместе, чеки `C002` — отдельно, чеки `C003` — в третий конверт.
Затем складываем суммы внутри каждого конверта.

| Клиент | Его покупки | Сумма |
|---|---|---|
| `C001` | 350 + 450 | 800 ₽ |
| `C002` | 200 | 200 ₽ |
| `C003` | 300 + 500 + 400 | 1 200 ₽ |

В pandas это два действия: **`groupby` собирает покупки одного клиента вместе, `sum` складывает их суммы**.
Применим команду ко всему файлу, а покажем первые три результата.

In [ ]:
total_by_client = purchases.groupby("client_id")["amount_rub"].sum()
total_by_client.head(3)

Прочитаем команду по частям:

| Запись | Что делаем |
|---|---|
| `groupby("client_id")` | Собираем в одну группу строки с одинаковым ID клиента |
| `["amount_rub"]` | Берём суммы покупок |
| `.sum()` | Складываем суммы отдельно внутри каждой группы |

Теперь **один результат относится к одному клиенту**, а не к одной покупке.
Для следующего шага превратим результат в таблицу: `reset_index()` вернёт ID клиента в обычный столбец.

In [ ]:
features = total_by_client.reset_index()
features.head(3)

## 2. Join: добавляем известный исход к сумме покупок

Хотим по расходам мая предсказать, будет ли клиент покупать в июне.
В `practice_target.csv` записан уже известный исход июня:

- **`target = 1`** — в июне покупок не было.
- **`target = 0`** — в июне покупки были.

Суммы мая известны до июня. Исход июня используем как правильный ответ для обучения и проверки.

**Левая таблица `target`:** список клиентов и правильных ответов.

In [ ]:
target = pd.read_csv("practice_target.csv")
target.head(3)

**Правая таблица `features`:** сумма покупок каждого клиента за май.

In [ ]:
features.head(3)

Нужно добавить сумму к **тому же самому клиенту**. Для этого в обеих таблицах есть `client_id`.

| Клиент слева | Что ищем справа | Что получится в одной строке |
|---|---|---|
| `C001`, target = 0 | Сумму для `C001` | `C001`, 0, 800 ₽ |
| `C002`, target = 1 | Сумму для `C002` | `C002`, 1, 200 ₽ |
| `C003`, target = 0 | Сумму для `C003` | `C003`, 0, 1 200 ₽ |

Соединение таблиц называют **join**. В pandas воспользуемся методом **`merge()`**:

In [ ]:
data = target.merge(features, on="client_id", how="left")
data.head(3)

`target` — таблица слева, `features` — справа.
`on="client_id"` означает «ищем одинаковые ID», а `how="left"` — «сохраняем всех клиентов слева».
Порядок строк справа не важен: соединение идёт по ID.

В этих файлах покупки есть у каждого клиента из `target`, поэтому пропусков после объединения нет.

## 3. Fit и predict: обучаем модель и получаем прогноз

Модель получит **один признак — сумму покупок мая**. Переведём рубли в тысячи рублей:
800 ₽ превратятся в 0,8 тыс. ₽. Информация остаётся той же, меняются только единицы.

In [ ]:
data["spending_thousand_rub"] = data["amount_rub"] / 1000
data[["client_id", "spending_thousand_rub", "target"]].head(3)

**X — что известно о клиенте до прогноза. y — какой исход хотим предсказать.**

В X берём только сумму в тысячах рублей. ID нужен для соединения таблиц, а target — правильный ответ;
их в признаки не включаем.

Разделим клиентов: на одних модель учится, других впервые увидит при проверке.
Готовую команду разделения можно использовать без изменений:

In [ ]:
from sklearn.model_selection import train_test_split

X = data[["spending_thousand_rub"]]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Клиентов для обучения:", len(X_train))
print("Клиентов для проверки:", len(X_test))

Получилось 18 клиентов для обучения и 6 для проверки.
`test_size=0.25` откладывает четверть клиентов; `random_state=42` делает разделение повторяемым;
`stratify=y` помогает сохранить соотношение классов.

**`fit(X_train, y_train)` — обучение.** Показываем модели признаки и правильные ответы вместе.
Логистическая регрессия сама подбирает параметры.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
print("Модель обучена")

**`predict(X_test)` — прогноз.** Теперь показываем только признаки других клиентов, без правильных ответов.
Модель возвращает классы 0 или 1.

In [ ]:
predictions = model.predict(X_test)
predictions

Сравним прогнозы с правильными ответами, которые хранились отдельно:

In [ ]:
comparison = pd.DataFrame({"Факт": y_test, "Прогноз": predictions})
comparison

## 4. Две метрики

**Accuracy** — доля всех правильных ответов.
Например, 5 совпадений из 6 дают accuracy около 0,83.

**Recall** — какую долю клиентов с фактом **1** модель нашла.
Например, если из двух клиентов без покупок модель нашла обоих, recall равен 1.

Сравниваем **`y_test` — факты** и **`predictions` — прогнозы** на тех же клиентах.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score

accuracy = accuracy_score(y_test, predictions)
recall = recall_score(y_test, predictions, zero_division=0)

print("Accuracy:", round(accuracy, 2))
print("Recall:", round(recall, 2))

Обе метрики принимают значения от 0 до 1. Смотрим на них вместе:
правильные ответы по частому классу могут скрывать пропуски клиентов класса 1.

`zero_division=0` задаёт результат 0 для recall, если в проверке вообще нет фактов класса 1.
В нашем разделении оба класса есть.

Путь от таблицы до прогноза: **покупки → `groupby` → признаки → `merge` с target → разделение клиентов → `fit` → `predict` → метрики**.
Модель обучалась только на `train`, а качество мы посчитали на `test`.